# 12 — Relevance review and grounded recommendations

This notebook demonstrates the evaluation **workflow**, not independent annotation. It uses Savoroute's fictional catalog and scripted mock reviewer decisions solely to test the export rules. For real evaluation, reviewers must annotate the frozen catalog independently; see `docs/relevance-workflow.md`.

In [ ]:
import json
from pathlib import Path

from savoroute.data.catalog import load_catalog
from savoroute.evaluation.annotations import prepare_relevance_workbook, split_reviewer_sheets, merge_reviewer_sheets, finalize_relevance_workbook
from savoroute.evaluation.retrieval import evaluate_search, parse_benchmark
from savoroute.retrieval.lexical import SearchFilters
from savoroute.services.discovery import DiscoveryService
from savoroute.services.recommendation import RecommendationPreferences

catalog = load_catalog("data/sample/catalog.json")
queries = json.loads(Path("data/sample/relevance_query_set_v1.json").read_text())
workbook = prepare_relevance_workbook(catalog, queries)
print(catalog.dataset_name, "synthetic=", catalog.synthetic)
print("queries:", len(workbook["queries"]), "eligible pairs:", len(workbook["judgments"]))
assert all(r["final_label"] is None for r in workbook["judgments"])

## Inspect the review queue

Each pair includes a canonical item ID. Reviewers should inspect the corresponding catalog entry and underlying source **without seeing model ranks**. Hard filters define the eligible universe. The workbook binds to a fingerprint of this frozen catalog.

In [ ]:
for row in workbook["judgments"][:5]:
    item = catalog.get(row["item_id"])
    print(row["case_id"], row["item_id"], item.title, "source:", item.source)
print("catalog fingerprint:", workbook["catalog_sha256"][:16], "...")

## Scripted reviewer files for a workflow demonstration

The two reviewer files are created independently from the blank template. The next cell **simulates** both reviews from the same fictional answer key; it does not represent independent human annotation. The merge preserves a deliberate disagreement for adjudication.

In [ ]:
mock_relevant = {
    "dev-tofu": {"p001"},
    "dev-pasta": {"p002"},
    "test-salad": {"p003"},
    "test-none": set(),
}
reviewer_a, reviewer_b = split_reviewer_sheets(catalog, workbook)
for sheet in (reviewer_a, reviewer_b):
    for row in sheet["judgments"]:
        positive = row["item_id"] in mock_relevant[row["case_id"]]
        row["relevant"] = positive
        row["note"] = "Programmed fictional positive label" if positive else ""

# Create one artificial disagreement; the merge does not decide it.
conflict_b = next(row for row in reviewer_b["judgments"]
                  if row["case_id"] == "dev-pasta" and row["item_id"] == "p002")
conflict_b["relevant"] = False
conflict_b["note"] = "Scripted counterexample"
merged = merge_reviewer_sheets(catalog, workbook, reviewer_a, reviewer_b)
assert sum(row["final_label"] is None for row in merged["judgments"]) == 1
conflict = next(row for row in merged["judgments"] if row["final_label"] is None)
conflict["final_label"] = True
conflict["adjudication_note"] = "Mock adjudicator selected the scripted positive label"

development, test, audit = finalize_relevance_workbook(catalog, merged)
print("Synthetic audit:", {key: audit[key] for key in
                           ("synthetic", "judged_pairs", "disagreements", "observed_agreement", "cohen_kappa")})
assert audit["synthetic"] and audit["disagreements"] == 1

## Evaluate the frozen test partition

Only complete judgments can be exported. The example below reports BM25 metrics for a **synthetic** test partition. Do not interpret them as real retrieval quality.

In [ ]:
judgments = parse_benchmark(test)
summary = evaluate_search(catalog.items, judgments, at_k=5, method="bm25")
print(summary.as_dict())
assert summary.at_k == 5

## Source-grounded recommendations

The recommendation service reuses catalog filtering and rank fusion, then applies a bounded preference bonus to already-retrieved items. It preserves source names, retrieval evidence, and structured catalog attributes; it does not invent restaurant availability or dietary safety claims.

In [ ]:
service = DiscoveryService(catalog)
result = service.recommend(
    "spicy tofu noodles",
    filters=SearchFilters(kind="recipe", dietary_tags=("vegan",)),
    preferences=RecommendationPreferences(cuisines=("Japanese",)),
    limit=3,
)
print("status:", result.status, "eligible:", result.eligible_items)
for row in result.results:
    print(row.item.item_id, row.item.title, row.item.source,
          "retrieval sources:", [entry.source for entry in row.evidence],
          "matched cuisines:", row.matched_cuisines)
assert result.results[0].item.item_id == "p001"
print(result.as_dict()["limitations"])

## Negative cases and boundaries

A restaurant with the tag `vegan-options` does not automatically satisfy a request for a verified `vegan` tag. Missing metadata fails closed for hard filters. The absence of a catalog match must not be described as the absence of a real-world option.

In [ ]:
not_verified = service.recommend(
    "Mediterranean", filters=SearchFilters(kind="restaurant", dietary_tags=("vegan",)),
)
print(not_verified.status, "matches:", len(not_verified.results))
assert not_verified.status == "no_eligible_items"
